# ODE lecture: executable figure source
Run all cells in order. MyST executes this notebook during the slide build.
Plots preserve the approved dimensionless formulas and linear reference maps in `figure-evidence.json`.
Near-square canvases are designed for a two-column projector layout.
The reference maps illustrate these linear test problems; they are not general ODE solver implementations.


In [ ]:
import io
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, ScalarFormatter
from IPython.display import SVG, display

BLUE, TEAL, ORANGE, PURPLE = '#2c5f9e', '#287c70', '#a56536', '#78629a'
plt.rcParams.update({
    'figure.figsize': (5.2, 4.5), 'figure.dpi': 150,
    'font.family': 'DejaVu Sans', 'font.size': 18,
    'axes.labelsize': 20, 'xtick.labelsize': 18, 'ytick.labelsize': 18,
    'legend.fontsize': 16, 'lines.linewidth': 2.5,
    'mathtext.fontset': 'cm', 'svg.fonttype': 'path',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#64748b', 'axes.labelcolor': '#243447',
    'xtick.color': '#243447', 'ytick.color': '#243447',
    'axes.prop_cycle': plt.cycler(color=[BLUE, TEAL, ORANGE, PURPLE]),
})

def canvas(xlabel, ylabel):
    fig, ax = plt.subplots()
    fig.subplots_adjust(left=.24, right=.96, bottom=.19, top=.89)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.yaxis.set_major_locator(MaxNLocator(4))
    ax.grid(alpha=.18, linewidth=.7)
    ax.set_axisbelow(True)
    return fig, ax

def emit(fig):
    # Keep the authored canvas; tight cropping would change its aspect ratio.
    buf = io.StringIO()
    fig.savefig(buf, format='svg', metadata={'Date': None})
    plt.close(fig)
    display(SVG(buf.getvalue()))

A = np.array([[0., 1.], [-1., 0.]])

def maps(h):
    # Polynomial of the oscillator's rotation generator, as in the original
    # reference figure script. Explicit scalar arithmetic preserves provenance.
    result = {}
    for name, degree in [('Euler', 1), ('Midpoint', 2), ('RK4', 4)]:
        v = sum((-1j*h)**k/math.factorial(k) for k in range(degree+1))
        result[name] = np.array([[v.real, -v.imag], [v.imag, v.real]])
    result['Leapfrog'] = np.array([[1-h*h/2, h], [-h*(1-h*h/4), 1-h*h/2]])
    return result

def trajectory(matrix, n):
    a,b,c,d = map(float, matrix.flat)
    q,p = 1.,0.
    states = [(q,p)]
    for i in range(n):
        q,p = a*q+b*p, c*q+d*p
        states.append((q,p))
    return np.array(states)

def phase_error(states, times):
    # Incremental unwrap used by the original frozen reference calculation.
    previous = 0.
    unwrapped = 0.
    result = []
    for i, (q,p) in enumerate(states):
        angle = math.atan2(-p, q)
        if i:
            unwrapped += (angle-previous+math.pi)%(2*math.pi)-math.pi
        previous = angle
        result.append(unwrapped-times[i])
    return np.array(result)

# Existing, frozen lecture experiment: 100 steps/period for 20 periods.
h = 2*np.pi/100
times = h*np.arange(2001)
paths = {name: trajectory(matrix, 2000) for name, matrix in maps(h).items()}
energy = {name: np.sum(state**2, axis=1)-1 for name, state in paths.items()}
phase = {name: phase_error(state, times) for name, state in paths.items()}


In [ ]:
s = np.linspace(0, 1.5, 301)
fig, ax = canvas(r'$rt$', r'$N/N_0$')
ax.plot(s, np.exp(s))
ax.scatter([0], [1], color=BLUE, zorder=3)
ax.set(xlim=(0, 1.5), ylim=(0, 5), xticks=[0, .5, 1, 1.5], yticks=[0, 1, 2, 3, 4, 5])
emit(fig)

In [ ]:
s = np.linspace(0, 1.5, 301)
fig, ax = canvas(r'$rt$', r'$N/N_0$')
ax.plot(s, np.exp(s))
for j in [1, 2]:
    x, y = j*np.log(2), 2**j
    ax.plot([x, x], [0, y], '--', color=TEAL, linewidth=1.4)
    ax.scatter([x], [y], color=TEAL, zorder=3)
ax.set(xlim=(0, 1.5), ylim=(0, 5), xticks=[0, np.log(2), 2*np.log(2)],
       xticklabels=[r'$0$', r'$\ln 2$', r'$2\ln 2$'], yticks=[0, 1, 2, 4])
emit(fig)

In [ ]:
s = np.linspace(0, 3, 301)
fig, ax = canvas(r'$\lambda t$', r'$N/N_0$ (log scale)')
ax.semilogy(s, np.exp(-s))
ax.scatter([0, np.log(2), 2*np.log(2)], [1, .5, .25], color=TEAL, zorder=3)
ax.set(xlim=(0, 3), ylim=(.04, 1.15), yticks=[.05, .1, .25, .5, 1], xticks=[0, 1, 2, 3])
ax.set_yticklabels([rf"${value:g}$" for value in [.05,.1,.25,.5,1]])
ax.minorticks_off()
emit(fig)

In [ ]:
s = np.linspace(0, 1.5, 301)
fig, ax = canvas(r'$|a|t$', r'$N/N_0$ (log scale)')
ax.semilogy(s, np.exp(s), label=r'$a>0$')
ax.semilogy(s, np.exp(-s), '--', label=r'$a<0$')
ax.set(xlim=(0, 1.5), ylim=(.2, 5), xticks=[0, .5, 1, 1.5], yticks=[.25, .5, 1, 2, 4])
fig.subplots_adjust(top=.79)
ax.legend(frameon=False, loc='lower center',bbox_to_anchor=(.5,1),ncol=2)
ax.set_yticklabels([rf"${value:g}$" for value in [.25,.5,1,2,4]])
ax.minorticks_off()
emit(fig)

In [ ]:
s = np.linspace(0, 4, 301)
fig, ax = canvas(r'$kt$', r'$\theta/\theta_0$ (log scale)')
ax.semilogy(s, np.exp(-s))
ax.set(xlim=(0, 4), ylim=(.01, 1.2), yticks=[.01, .1, 1])
emit(fig)

In [ ]:
z = np.linspace(0, 1, 101)
fig, ax = canvas(r'$z/H$', r'$(P-P_{\rm surface})/(\rho gH)$')
ax.plot(z, z)
ax.set(xlim=(0, 1), ylim=(0, 1.1), xticks=[0, .5, 1], yticks=[0, .5, 1])
emit(fig)

In [ ]:
x = np.linspace(-4, 4, 501)
fig, ax = canvas(r'$x$ (scaled)', r'Temperature excess $\theta$')
for t, style in zip([0, .5, 1], ['-', '--', ':']):
    theta = np.exp(-x*x/(1+4*t))/np.sqrt(1+4*t)
    ax.plot(x, theta, style, label=rf'$t={t:g}$')
ax.set(xlim=(-4, 4), ylim=(0, 1.1), xticks=[-4, -2, 0, 2, 4], yticks=[0, .5, 1])
ax.legend(frameon=False, ncol=3, loc='lower center', bbox_to_anchor=(.5, 1),
          handlelength=1, columnspacing=.7, fontsize=16)
emit(fig)

In [ ]:
x = np.linspace(0, 1.2, 301)
fig, ax = canvas(r'$x$', r'$y$')
xx, yy = np.meshgrid(np.linspace(0, 1.2, 9), np.linspace(0, 3.2, 9))
# Direction segments drawn in data coordinates: slope dy/dx = y.
for u, v in zip(xx.flat, yy.flat):
    d=.025
    ax.plot([u-d, u+d], [v-v*d, v+v*d], color='#b5c1cc', linewidth=1)
ax.plot(x, .5*np.exp(x), label=r'$y(0)=0.5$')
ax.plot(x, np.exp(x), '--', label=r'$y(0)=1$')
ax.set(xlim=(0, 1.2), ylim=(0, 4))
ax.legend(frameon=False, loc='upper left')
emit(fig)

In [ ]:
t = np.linspace(0, .4, 301)
fig, ax = canvas(r'$t$', r'$u$')
ax.plot(t, np.exp(-t), label='Exact')
ax.plot(t, 1-t, '--', label='Initial tangent')
ax.scatter([.2], [.8], color=TEAL, zorder=3)
ax.set(xlim=(0, .4), ylim=(.55, 1.04), xticks=[0, .2, .4])
ax.legend(frameon=False, loc='lower left')
emit(fig)

In [ ]:
t = np.linspace(0, .24, 301)
fig, ax = canvas(r'$t$', r'$u$')
ax.plot(t, np.exp(-t), label='Exact')
ax.plot([0, .2], [1, .8], '--o', label='Euler')
ax.scatter([.2], [np.exp(-.2)], color=BLUE, zorder=3)
ax.set(xlim=(0, .24), ylim=(.76, 1.03), xticks=[0, .1, .2])
ax.legend(frameon=False, loc='upper right')
emit(fig)

In [ ]:
t = np.linspace(0, 9, 501)
fig, ax = canvas(r'$t$', r'$u$')
ax.plot(t, np.exp(-t), label='Exact')
ax.plot(1.5*np.arange(7), (-.5)**np.arange(7), '--o', label='Euler')
ax.axhline(0, color='#64748b', linewidth=1)
ax.set(xlim=(0, 9), ylim=(-.65, 1.15), xticks=[0, 3, 6, 9])
ax.legend(frameon=False, loc='upper right')
emit(fig)

In [ ]:
# Same decay model and the step sizes already introduced on this slide.
# Compare magnitudes at a common endpoint; retain the signed preceding panel.
t=np.linspace(0,9,501)
fig,ax=canvas(r'$t$',r'$|u|$ (log scale)')
fig.subplots_adjust(top=.79)
ax.semilogy(t,np.exp(-t),color=BLUE,label='Exact')
for step,color,style in [(.25,TEAL,'--'),(1.5,ORANGE,'-.'),(3,PURPLE,':')]:
    nodes=step*np.arange(round(9/step)+1)
    values=(1-step)**np.arange(len(nodes))
    ax.semilogy(nodes,np.abs(values),style,color=color,
                marker='o',markersize=3 if step==.25 else 6,label=rf'$h={step:g}$')
    np.testing.assert_allclose(np.diff(values),-step*values[:-1],rtol=1e-12,atol=0)
    assert nodes[-1]==9
ax.set(xlim=(0,9),ylim=(1e-5,12),xticks=[0,3,6,9],yticks=[1e-5,1e-3,1e-1,1e1])
ax.legend(frameon=False,loc='lower center',bbox_to_anchor=(.5,1),ncol=2)
emit(fig)


In [ ]:
t = np.linspace(0, 1, 301)
fig, ax = canvas(r'$t$', r'$u$')
ax.plot(t, np.exp(-t), label='Exact')
ax.plot(.2*np.arange(6), .8**np.arange(6), '--o', label='Euler')
ax.set(xlim=(0, 1), ylim=(.28, 1.05), xticks=[0, .5, 1])
ax.annotate('',xy=(1,np.exp(-1)),xytext=(1,.8**5),
            arrowprops=dict(arrowstyle='<->',color=PURPLE,lw=1.6))
ax.legend(frameon=False, loc='lower left')
emit(fig)

In [ ]:
fig, ax = canvas(r'$x$', '')
for y, n, label, color in [(1, 4, r'$h$', BLUE), (0, 8, r'$h/2$', TEAL)]:
    x=np.linspace(0, 1, n+1)
    ax.plot(x, np.full_like(x,y), 'o-', color=color)
ax.set(xlim=(-.05, 1.05), ylim=(-.5, 1.5), yticks=[0, 1],
       yticklabels=[r'$h/2$', r'$h$'], xticks=[0, .5, 1])
ax.spines['left'].set_visible(False)
ax.grid(False)
emit(fig)

In [ ]:
x = np.linspace(0, 2, 101)
fig, ax = canvas(r'Displacement $x$', r'$F/k$')
ax.plot(x, -x)
ax.set(xlim=(0, 2), ylim=(-2.2, .2), xticks=[0, 1, 2], yticks=[-2, -1, 0])
emit(fig)

In [ ]:
t = np.linspace(0, 2*np.pi, 501)
fig, ax = canvas(r'$t$ (scaled)', r'State $(q,p)$')
ax.plot(t, np.cos(t), label=r'$q=\cos t$')
ax.plot(t, -np.sin(t), '--', label=r'$p=-\sin t$')
ax.set(xlim=(0, 2*np.pi), ylim=(-1.25, 1.55),
       xticks=[0, np.pi, 2*np.pi], xticklabels=[r'$0$', r'$\pi$', r'$2\pi$'],
       yticks=[-1, 0, 1])
ax.legend(frameon=False, loc='upper center', ncol=2, handlelength=1, columnspacing=.6, fontsize=16)
emit(fig)

In [ ]:
t = np.linspace(0, 2*np.pi, 501)
fig, ax = canvas(r'$q$', r'$p$')
ax.plot(np.cos(t), -np.sin(t))
ax.scatter([1, 0], [0, -1], color=TEAL, zorder=3)
ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), xticks=[-1,0,1], yticks=[-1,0,1])
ax.set_aspect('equal', adjustable='box')
emit(fig)

In [ ]:
# Reuse the existing Euler phase experiment for every method.
phase_h, phase_steps = .1, 120
phase_times = phase_h*np.arange(phase_steps+1)
phase_paths = {name: trajectory(maps(phase_h)[name], phase_steps)
               for name in ['Euler', 'Midpoint', 'RK4']}
phase_styles = {'Euler': ('#90969d', '--', 'Euler'),
                'Midpoint': ('#007567', (0,(5,5)), 'RK2'),
                'RK4': ('#7130b0', (5,(5,5)), 'RK4')}

def phase_comparison(names):
    fig, ax = canvas(r'$q$', r'$p$')
    fig.subplots_adjust(top=.79)
    t = np.linspace(0, 2*np.pi, 501)
    ax.plot(np.cos(t),-np.sin(t),color='#26313d',linewidth=1.3,
            linestyle=(0,(1,5)),label='Exact',zorder=4)
    for name in names:
        state=phase_paths[name]
        color, style, label = phase_styles[name]
        ax.plot(state[:,0],state[:,1],linestyle=style,color=color,label=label,
                linewidth=1.4 if name=='Euler' else 3.8,
                alpha=(.65 if len(names)==1 else .30) if name=='Euler' else 1,
                zorder=1 if name=='Euler' else 3)
        if name!='Euler':
            ax.plot(*state[-1],marker='s' if name=='Midpoint' else '^',
                    markersize=11 if name=='Midpoint' else 8,
                    markerfacecolor='white',markeredgecolor=color,
                    markeredgewidth=2,linestyle='none',zorder=6)
    extent=2 if len(names)==1 else 1.08
    ax.set(xlim=(-extent,extent),ylim=(-extent,extent),
           xticks=[-2,0,2] if len(names)==1 else [-1,0,1],
           yticks=[-2,0,2] if len(names)==1 else [-1,0,1])
    ax.set_aspect('equal',adjustable='box')
    ax.legend(frameon=False,loc='lower center',bbox_to_anchor=(.5,1),ncol=2)
    emit(fig)

# Independently derived one-step energy multipliers for this linear oscillator.
gains={'Euler':1+phase_h**2, 'Midpoint':1+phase_h**4/4,
       'RK4':1-phase_h**6/72+phase_h**8/576}
for name, states in phase_paths.items():
    expected=.5*gains[name]**np.arange(phase_steps+1)
    np.testing.assert_allclose(.5*np.sum(states**2,axis=1),expected,
                               rtol=1e-12,atol=1e-12)


In [ ]:
phase_comparison(['Euler'])


In [ ]:
phase_comparison(['Euler', 'Midpoint'])


In [ ]:
phase_comparison(['Euler', 'Midpoint', 'RK4'])


In [ ]:
# A magnified phase-plane endpoint view; same experiment, no new step size.
exact_end=np.array([np.cos(phase_times[-1]),-np.sin(phase_times[-1])])
rk2_end=phase_paths['Midpoint'][-1]
center=(exact_end+rk2_end)/2
halfwidth=.9*np.max(np.abs(exact_end-rk2_end))
fig, ax=canvas(r'$q$', r'$p$')
fig.subplots_adjust(top=.80)
for name, marker in [('Midpoint','s'),('RK4','^')]:
    states=phase_paths[name]
    color, style, label=phase_styles[name]
    if name=='Midpoint':
        ax.annotate('',xy=states[-1],xytext=exact_end,
                    arrowprops=dict(arrowstyle='->',color=color,lw=1.8),zorder=3)
    ax.plot(*states[-1],marker,markeredgecolor=color,markerfacecolor='white',
            markeredgewidth=2.5,markersize=11,label=label,zorder=5)
t=np.linspace(0,phase_times[-1],4001)
ax.plot(np.cos(t),-np.sin(t),color='#26313d',linewidth=1.5,linestyle='--')
ax.plot(*exact_end,'o',color='#26313d',markersize=4,label='Exact',zorder=8)
ax.set(xlim=(center[0]-halfwidth,center[0]+halfwidth),
       ylim=(center[1]-halfwidth,center[1]+halfwidth))
ax.set_aspect('equal',adjustable='box')
ax.xaxis.set_major_locator(MaxNLocator(3))
ax.yaxis.set_major_locator(MaxNLocator(3))
fig.legend(frameon=False,loc='upper center',bbox_to_anchor=(.5,.98),ncol=3)
emit(fig)


In [ ]:
# A magnified phase-plane endpoint view; same experiment, no new step size.
exact_end=np.array([np.cos(phase_times[-1]),-np.sin(phase_times[-1])])
rk2_end=phase_paths['Midpoint'][-1]
center=(exact_end+rk2_end)/2
halfwidth=.9*np.max(np.abs(exact_end-rk2_end))
fig, ax=canvas(r'$q$', r'$p$')
fig.subplots_adjust(top=.80)
for name, marker in [('Midpoint','s')]:
    states=phase_paths[name]
    color, style, label=phase_styles[name]
    if name=='Midpoint':
        ax.annotate('',xy=states[-1],xytext=exact_end,
                    arrowprops=dict(arrowstyle='->',color=color,lw=1.8),zorder=3)
    ax.plot(*states[-1],marker,markeredgecolor=color,markerfacecolor='white',
            markeredgewidth=2.5,markersize=11,label=label,zorder=5)
t=np.linspace(0,phase_times[-1],4001)
ax.plot(np.cos(t),-np.sin(t),color='#26313d',linewidth=1.5,linestyle='--')
ax.plot(*exact_end,'o',color='#26313d',markersize=4,label='Exact',zorder=8)
ax.set(xlim=(center[0]-halfwidth,center[0]+halfwidth),
       ylim=(center[1]-halfwidth,center[1]+halfwidth))
ax.set_aspect('equal',adjustable='box')
ax.xaxis.set_major_locator(MaxNLocator(3))
ax.yaxis.set_major_locator(MaxNLocator(3))
fig.legend(frameon=False,loc='upper center',bbox_to_anchor=(.5,.98),ncol=3)
emit(fig)


In [ ]:
# Coordinates relative to the exact endpoint expose the small RK4 error.
# Both axes have the same microunit scale; this remains a phase-plane zoom.
exact_end=np.array([np.cos(phase_times[-1]),-np.sin(phase_times[-1])])
rk4_offset=(phase_paths['RK4'][-1]-exact_end)*1e6
fig, ax=canvas(r'$10^6(q-q_{\mathrm{exact}}(T))$',
               r'$10^6(p-p_{\mathrm{exact}}(T))$')
fig.subplots_adjust(left=.24,bottom=.24,top=.79)
near_t=np.linspace(phase_times[-1]-2e-5,phase_times[-1]+2e-5,401)
exact_curve=np.column_stack([np.cos(near_t),-np.sin(near_t)])
ax.plot(*(1e6*(exact_curve-exact_end)).T,color='#26313d')
ax.plot(0,0,'o',color='#26313d',markersize=8,label='Exact')
ax.plot(*rk4_offset,'^',color=phase_styles['RK4'][0],markersize=10,label='RK4')
ax.annotate('',xy=rk4_offset,xytext=(0,0),
            arrowprops=dict(arrowstyle='->',color=phase_styles['RK4'][0],lw=1.5))
ax.set(xlim=(-12,12),ylim=(-12,12),xticks=[-10,0,10],yticks=[-10,0,10])
ax.set_aspect('equal',adjustable='box')
ax.legend(frameon=False,loc='lower center',bbox_to_anchor=(.5,1),ncol=2)
emit(fig)


In [ ]:
# The existing midpoint example: show differences too small for the state plot.
coordinates=np.array([.05,.1])
computed=np.array([1.05,1.105])
absolute_errors=np.abs(computed-np.exp(coordinates))
fig,ax=canvas(r'$x$',r'$|y_{\mathrm{approx}}-y_{\mathrm{exact}}|$')
fig.subplots_adjust(top=.79)
for x,error,color,marker,label in zip(coordinates,absolute_errors,
        [TEAL,ORANGE],['o','s'],['Trial midpoint','Final update']):
    ax.scatter([x],[error],color=color,marker=marker,s=85,label=label,zorder=3)
ax.set_yscale('log')
ax.set(xlim=(0,.12),ylim=(1e-5,3e-3),xticks=[0,.05,.1],yticks=[1e-5,1e-4,1e-3])
ax.minorticks_off()
ax.legend(frameon=False,loc='lower center',bbox_to_anchor=(.5,1))
emit(fig)


In [ ]:
hs = .1/2**np.arange(5)
errors={}
fig, ax = canvas(r'Step size $h$', r'Endpoint state error')
for name, style in zip(['Euler', 'Midpoint', 'RK4'], ['o-', 's--', '^:']):
    errors[name]=np.array([np.max(np.abs(trajectory(maps(step)[name], round(2/step))[-1]
                                      -[np.cos(2), -np.sin(2)])) for step in hs])
    ax.loglog(hs, errors[name], style, label='RK2' if name=='Midpoint' else name)
ax.set(xticks=[.00625, .025, .1], xticklabels=[r'$0.00625$', r'$0.025$', r'$0.1$'],
       yticks=[1e-10,1e-7,1e-4,1e-1], ylim=(1e-11,.2))
ax.minorticks_off()
ax.legend(frameon=False, loc='upper left')
emit(fig)

In [ ]:
values=[.8, .81, .82, np.exp(-.2)]
labels=['Full step', 'Half steps', 'Richardson', 'Exact']
fig, ax=canvas(r'Endpoint $u(0.2)$', '')
for j, (value, color) in enumerate(zip(values, [BLUE,TEAL,ORANGE,PURPLE])):
    ax.scatter([value], [3-j], s=80, color=color, zorder=3)
ax.set(xlim=(.794,.825), ylim=(-.5,3.5), yticks=[3,2,1,0],
       yticklabels=labels, xticks=[.80,.81,.82])
# Categorical labels need more left margin, rather than overlapping markers.
fig.subplots_adjust(left=.35, right=.96)
ax.tick_params(axis='y', labelsize=17)
ax.spines['left'].set_visible(False)
ax.grid(axis='y', visible=False)
emit(fig)

In [ ]:
# Logarithms require positive values: plot magnitude, omit exact zeros.
fig, ax = canvas('Periods', r'$|E-E_0|/E_0$')
fig.subplots_adjust(top=.79)
for name, color, style, label in [('Midpoint',TEAL,'-','RK2'),
                                  ('Leapfrog',ORANGE,'--','Leapfrog'),
                                  ('RK4',PURPLE,':','RK4')]:
    magnitude=np.abs(energy[name])
    ax.semilogy(times/(2*np.pi), np.ma.masked_equal(magnitude,0),
                style, color=color, label=label,
                linewidth=1.2 if name=='Leapfrog' else 3,
                alpha=.55 if name=='Leapfrog' else 1,
                zorder=4 if name=='RK4' else 2)
ax.set(xlim=(0,20), ylim=(1e-10,2e-2), xticks=[0,10,20],
       yticks=[1e-10,1e-8,1e-6,1e-4,1e-2])
ax.legend(frameon=False,loc='lower center',bbox_to_anchor=(.5,1),ncol=2)
emit(fig)


In [ ]:
fig, ax = canvas('Periods', r'$(E-E_0)/E_0$')
ax.plot(times/(2*np.pi), energy['RK4'], color=PURPLE, label='RK4, magnified')
ax.set(xlim=(0,20), ylim=(-2e-6,0), xticks=[0,10,20], yticks=[-2e-6,-1e-6,0])
fmt=ScalarFormatter(useMathText=True)
fmt.set_powerlimits((0,0))
ax.yaxis.set_major_formatter(fmt)
ax.legend(frameon=False, loc='lower left', fontsize=16)
emit(fig)

In [ ]:
fig, ax = canvas('Periods', r'$|\phi-\phi_{\mathrm{exact}}|$ (rad)')
fig.subplots_adjust(top=.79)
for name, color, style, label in [('Midpoint',TEAL,'-','RK2'),
                                  ('Leapfrog',ORANGE,'--','Leapfrog'),
                                  ('RK4',PURPLE,':','RK4')]:
    magnitude=np.abs(phase[name])
    ax.semilogy(times/(2*np.pi), np.ma.masked_equal(magnitude,0),
                style,color=color,label=label)
ax.set(xlim=(0,20),ylim=(1e-8,2e-1),xticks=[0,10,20],
       yticks=[1e-8,1e-6,1e-4,1e-2])
ax.legend(frameon=False,loc='lower center',bbox_to_anchor=(.5,1),ncol=2)
emit(fig)


In [ ]:
s = np.linspace(0, 30, 501)
fig, ax = canvas(r'$1000t$', r'$u$ (log scale)')
ax.semilogy(s, np.exp(-s), label='Exact')
ax.semilogy(10*np.arange(4), 11.**(-np.arange(4)), '--o', label='Backward Euler')
ax.set(xlim=(0,30), ylim=(1e-14,2), xticks=[0,10,20,30], yticks=[1,1e-7,1e-14])
ax.minorticks_off()
ax.legend(frameon=False, loc='lower left', fontsize=16)
emit(fig)

In [ ]:
# Check the unchanged lecture experiments against the stored reference evidence.
import json
from pathlib import Path
reference = json.loads(Path('figure-evidence.json').read_text())
for name in ['Euler', 'Midpoint', 'RK4']:
    np.testing.assert_allclose(errors[name][:-1]/errors[name][1:],
                               reference['order_ratios'][name], rtol=1e-5, atol=0)
for name in ['Midpoint', 'RK4', 'Leapfrog']:
    np.testing.assert_allclose(phase[name][-1],
        reference['geometry_results'][name]['final_phase_error'], rtol=1e-7, atol=1e-12)
    np.testing.assert_allclose(np.max(np.abs(energy[name])),
        reference['geometry_results'][name]['max_abs_energy_error'], rtol=1e-7, atol=1e-12)
print('Reference maps, endpoint orders, and oscillator energy/phase match the frozen lecture evidence.')
